In [ ]:
pip install transformers torch datasets scikit-learn pandas

**Loading The dataset**

In [ ]:
from datasets import load_dataset
import pandas as pd

df = load_dataset('Tobi-Bueck/customer-support-tickets')

In [ ]:
df = pd.DataFrame(df['train'])
df.head()

In [ ]:
print(df.columns)

In [ ]:
# target label
df['text'] = df['subject'] + ' ' + df['body']
df['queue'].value_counts()


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['category'] = le.fit_transform(df['queue'])

In [ ]:
df['category'].unique()

**Train test data splitting**

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['category'])

**Tokenization**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
  return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=128)



In [ ]:
test_df['text'].head()

**Dataset Formatting**

In [ ]:
from datasets import Dataset

train_df['text'] = train_df['subject'].fillna('') + ' ' + train_df['body'].fillna('')
test_df['text'] = test_df['subject'].fillna('') + ' ' + test_df['body'].fillna('')

train_dataset = Dataset.from_pandas(train_df[['text','category']],preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[['text','category']],preserve_index=False)

train_dataset = train_dataset.map(tokenize, batched=True, batch_size=len(train_dataset))
test_dataset = test_dataset.map(tokenize, batched=True, batch_size=len(test_dataset))

train_dataset = train_dataset.rename_columns({'category':'labels'})
test_dataset = test_dataset.rename_columns({'category':'labels'})

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

**Model Loading**

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(le.classes_))

**Training & Evaluation**

In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score,f1_score

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='no',
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    save_strategy='no'

)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }


trainer  = Trainer(model=model, args=training_args, train_dataset=train_dataset.select(range(30000)), eval_dataset=test_dataset.select(range(12000)),compute_metrics=compute_metrics)
trainer.train()

**Evaluation**

In [ ]:
import torch
def predict_ticket(text):
    inputs = tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    # Move inputs to the same device as the model
    device = model.device # Get the device of the model
    inputs = {k: v.to(device) for k, v in inputs.items()} # Move all input tensors to the device

    with torch.no_grad():
        logits = model(**inputs).logits
    prediction = torch.argmax(logits, dim=1).item()
    return prediction

In [ ]:
print(predict_ticket("my account is not working"))
print(predict_ticket("payment related issue"))
print(predict_ticket("i want to cancel my subscription"))
print(predict_ticket("damaged package got"))

In [ ]:
res = trainer.evaluate()
print(res)

In [ ]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(label_mapping)